In [57]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", module="IPython")

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    # Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
    if root_dir.parts[-1:] == ('airquality',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

print(f"Root dir: {root_dir}")

# Add the root directory to the `PYTHONPATH` 
if root_dir not in sys.path:
    sys.path.append(root_dir)
    print(f"Added the following directory to the PYTHONPATH: {root_dir}")

# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Root dir: /Users/yuxinjin/Desktop/KTH/Second-P2/SML/Lab1
HopsworksSettings initialized!


<span style="font-width:bold; font-size: 3rem; color:#333;">- Part 01: Feature Backfill for Air Quality Data</span>


## 🗒️ You have the following tasks
1. Choose an Air Quality Sensor
2. Update the country, city, and street information to point to YOUR chosen Air Quality Sensor
3. Download historical measures for your Air Quality Sensor as a CSV file
4. Update the path of the CSV file in this notebook to point to the one that you downloaded
5. Create an account on www.hopsworks.ai and get your HOPSWORKS_API_KEY
6. Run this notebook



### <span style='color:#ff5f27'> 📝 Imports

In [58]:
import datetime
import requests
import pandas as pd
import hopsworks
from mlfs.airquality import util
import datetime
from pathlib import Path
import json
import re
import os
import warnings
warnings.filterwarnings("ignore")

---

## Hopsworks API Key
You need to have registered an account on app.hopsworks.ai.

Save the HOPSWORKS_API_KEY  to ~/.env file in the root directory of your project

 * mv .env.example .env
 * edit .env

In the .env file, update HOPSWORKS_API_KEY:

`HOPSWORKS_API_KEY="put API KEY value in this string"`


In [59]:
import hopsworks

conn = hopsworks.connection(
    api_key_value=os.getenv("HOPSWORKS_API_KEY"),
    host=os.getenv("HOPSWORKS_HOST")
)

projects = conn.get_projects()
for p in projects:
    print("-", p.name)


2025-12-17 16:27:54,389 INFO: Closing external client and cleaning up certificates.
2025-12-17 16:27:54,391 INFO: Initializing external client
2025-12-17 16:27:54,391 INFO: Base URL: https://c.app.hopsworks.ai:443


- Lab1YuxinLeah


In [60]:
# project = hopsworks.login()

from dotenv import load_dotenv
import os
import hopsworks
load_dotenv()  

project = hopsworks.login(
    project="Lab1YuxinLeah",
    api_key_value=os.getenv("HOPSWORKS_API_KEY"),
    host=os.getenv("HOPSWORKS_HOST")
)

print("Logged into project:", project.name)


2025-12-17 16:27:56,359 INFO: Closing external client and cleaning up certificates.
Connection closed.
2025-12-17 16:27:56,360 INFO: Initializing external client
2025-12-17 16:27:56,360 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-12-17 16:27:59,652 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1286353
Logged into project: Lab1YuxinLeah


## <span style='color:#ff5f27'> 🌍 Fetch historical data, save to CVS and create DataFrames </span>

## EU_SEK exchange rate historical data


In [61]:
today = datetime.date.today()
startDate="2015-08-30"
endDate = today.isoformat()
base_currency = "EUR"  # Use EUR as base currency
target_currency = "SEK"  # Target currency SEK

# Fetch data API url
url = f"https://api.frankfurter.dev/v1/{startDate}..{endDate}?base={base_currency}&symbols={target_currency}"

response = requests.get(url)
data = response.json()

# Convert JSON to DataFrame
df_exchange_rate = pd.DataFrame.from_dict(data["rates"], orient="index")

# Add a 'Date' column as the first column
df_exchange_rate.index.name = "Date"
df_exchange_rate.reset_index(inplace=True)

# Save to CSV
df_exchange_rate.to_csv("data/exchangeRates.csv", index=False)

print("CSV file saved as '../data/exchangeRates.csv'")

CSV file saved as '../data/exchangeRates.csv'


## EU Inflation rate

In [62]:
import io
START_DATE = "2015-09"
END_DATE   = "2025-12"
CACHE_PATH = "data/eu_inflation.csv"

EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
    "prc_hicp_manr/M.RCH_A.CP00.EU27_2020"
)

PARAMS = {
    "startPeriod": START_DATE,
    "endPeriod": END_DATE,
    "format": "SDMX-CSV"
}


if os.path.exists(CACHE_PATH):
    print("Loading EU inflation from cache")
    df_eu_inflation = pd.read_csv(CACHE_PATH)

else:
    print("Fetching EU inflation from Eurostat...")

    response = requests.get(EUROSTAT_URL, params=PARAMS, timeout=120)
    response.raise_for_status()

    df_raw = pd.read_csv(io.StringIO(response.text))

    df_eu_inflation = (
        df_raw[["TIME_PERIOD", "OBS_VALUE"]]
        .rename(columns={
            "TIME_PERIOD": "date",
            "OBS_VALUE": "eu_inflation_rate"
        })
        .dropna()
        .sort_values("date")
    )

    df_eu_inflation["date"] = pd.to_datetime(df_eu_inflation["date"] + "-01")

    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    df_eu_inflation.to_csv(CACHE_PATH, index=False)

    print(f"Saved {len(df_eu_inflation)} EU inflation records")


print(df_eu_inflation.head())

Loading EU inflation from cache
         date  eu_inflation_rate
0  2015-09-01                0.1
1  2015-10-01                0.3
2  2015-11-01                0.1
3  2015-12-01                0.2
4  2016-01-01                0.3


## Sweden inflation rate


In [63]:
START_DATE = "2015-09"  
END_DATE   = "2025-12"  
CACHE_PATH = "data/sweden_inflation.csv"

EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
    "prc_hicp_manr/M.RCH_A.CP00.SE"
)

PARAMS = {
    "startPeriod": START_DATE,
    "endPeriod": END_DATE,
    "format": "SDMX-CSV"
}


if os.path.exists(CACHE_PATH):
    print("✓ Loading Sweden inflation from cache")
    df_sweden_inflation = pd.read_csv(CACHE_PATH)

else:

    response = requests.get(EUROSTAT_URL, params=PARAMS, timeout=120)
    response.raise_for_status()
    df_raw = pd.read_csv(io.StringIO(response.text))

    df_sweden_inflation = (
        df_raw[["TIME_PERIOD", "OBS_VALUE"]]
        .rename(columns={
            "TIME_PERIOD": "date",
            "OBS_VALUE": "sweden_inflation_rate"
        })
        .dropna()
        .sort_values("date")
    )

    # Convert YYYY-MM → datetime
    df_sweden_inflation["date"] = df_sweden_inflation.to_datetime(df_sweden_inflation["date"] + "-01")

    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    df_sweden_inflation.to_csv(CACHE_PATH, index=False)

    print(f"✓ Saved {len(df_sweden_inflation)} rows to cache")

print(df_sweden_inflation.head())
print(f"\nRecords: {len(df_sweden_inflation)}")

✓ Loading Sweden inflation from cache
         date  sweden_inflation_rate
0  2015-09-01                    0.9
1  2015-10-01                    0.9
2  2015-11-01                    0.8
3  2015-12-01                    0.7
4  2016-01-01                    1.3

Records: 122


## Sweden interest rate

In [64]:
url = "https://api.riksbank.se/swestr/v1/SWESTR"
params = {
    "fromDate": "2015-09-01T00:00:00Z",
    "toDate": "2025-12-31T00:00:00Z"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

df_sweden_interest = pd.DataFrame(data)
df_sweden_interest["date"] = pd.to_datetime(df_sweden_interest["date"])
df_sweden_interest["rate"] = df_sweden_interest["rate"].astype(float)

df_sweden_interest = df_sweden_interest.sort_values("date").reset_index(drop=True)

# save to csv
df_sweden_interest.to_csv("data/sweden_interest_rate.csv", index=False)
print(df_sweden_interest.head())

    rate       date  pctl12_5  pctl87_5   volume  alternativeCalculation  \
0 -0.104 2021-09-01     -0.20       0.0  32942.0                   False   
1 -0.096 2021-09-02     -0.15       0.0  30762.0                   False   
2 -0.101 2021-09-03     -0.20       0.0  29849.0                   False   
3 -0.109 2021-09-06     -0.20       0.0  31594.0                   False   
4 -0.104 2021-09-07     -0.20       0.0  27221.0                   False   

  alternativeCalculationReason       publicationTime  republication  \
0                         None  2021-09-02T09:00:00Z          False   
1                         None  2021-09-03T09:00:00Z          False   
2                         None  2021-09-06T09:00:00Z          False   
3                         None  2021-09-07T09:00:00Z          False   
4                         None  2021-09-08T09:00:00Z          False   

   numberOfTransactions  numberOfAgents  
0                  39.0             6.0  
1                  40.0         

## <span style='color:#ff5f27'> 🌍 STEP 6: Data cleaning</span>


### Clean and prepare Exchange Rate, Inflation, and Interest Rate DataFrames

We need to clean and prepare three types of data:
1. **Exchange Rate**: `date` and `eur_sek_rate` columns
2. **Inflation**: `date`, `country`, and `inflation_rate` columns  
3. **Interest Rate**: `date`, `country`, and `interest_rate` columns

### Check the data types for the columns in your DataFrames

 * `date` should be of type datetime64[ns] 
 * Exchange rate, inflation, and interest rate should be of type float32

In [65]:
# Clean Exchange Rate Data
# Rename Date to date and SEK to eur_sek_rate
df_exchange_rate_clean = df_exchange_rate.copy()
df_exchange_rate_clean.rename(columns={'Date': 'date', 'SEK': 'eur_sek_rate'}, inplace=True)
df_exchange_rate_clean['date'] = pd.to_datetime(df_exchange_rate_clean['date'])
df_exchange_rate_clean['eur_sek_rate'] = df_exchange_rate_clean['eur_sek_rate'].astype('float32')
df_exchange_rate_clean = df_exchange_rate_clean[['date', 'eur_sek_rate']]

print("Exchange Rate Data:")
print(df_exchange_rate_clean.head())
print(f"\nShape: {df_exchange_rate_clean.shape}")
print(f"\nData types:\n{df_exchange_rate_clean.dtypes}")

Exchange Rate Data:
        date  eur_sek_rate
0 2015-08-28        9.4953
1 2015-08-31        9.5032
2 2015-09-01        9.4954
3 2015-09-02        9.4968
4 2015-09-03        9.3907

Shape: (2640, 2)

Data types:
date            datetime64[ns]
eur_sek_rate           float32
dtype: object


In [66]:
# Ensure datetime
df_eu_inflation_clean = df_eu_inflation.copy()
df_eu_inflation_clean["date"] = pd.to_datetime(df_eu_inflation_clean["date"])
df_eu_inflation_clean["eu_inflation_rate"] = (
    df_eu_inflation_clean["eu_inflation_rate"].astype("float32")
)
df_eu_inflation_clean = df_eu_inflation_clean[["date", "eu_inflation_rate"]]

df_sweden_inflation_clean = df_sweden_inflation.copy()
df_sweden_inflation_clean["date"] = pd.to_datetime(df_sweden_inflation_clean["date"])
df_sweden_inflation_clean["sweden_inflation_rate"] = (
    df_sweden_inflation_clean["sweden_inflation_rate"].astype("float32")
)
df_sweden_inflation_clean = df_sweden_inflation_clean[
    ["date", "sweden_inflation_rate"]
]

# Merge EU + Sweden inflation on date
df_inflation_clean = df_eu_inflation_clean.merge(
    df_sweden_inflation_clean,
    on="date",
    how="outer"
).sort_values("date").reset_index(drop=True)

print(df_inflation_clean.head())
print(df_inflation_clean.dtypes)


        date  eu_inflation_rate  sweden_inflation_rate
0 2015-09-01                0.1                    0.9
1 2015-10-01                0.3                    0.9
2 2015-11-01                0.1                    0.8
3 2015-12-01                0.2                    0.7
4 2016-01-01                0.3                    1.3
date                     datetime64[ns]
eu_inflation_rate               float32
sweden_inflation_rate           float32
dtype: object


### Clean Interest Rate Data

Note: You may need to load EU interest rate data separately if available. For now, we'll prepare Sweden interest rate data.

In [67]:
# Clean Sweden interest rate data (denormalized)
df_interest_clean = df_sweden_interest.copy()

df_interest_clean["date"] = pd.to_datetime(df_interest_clean["date"])

df_interest_clean = (
    df_interest_clean
        .rename(columns={"rate": "sweden_interest_rate"})
        .astype({"sweden_interest_rate": "float32"})
        [["date", "sweden_interest_rate"]]
        .sort_values("date")
        .reset_index(drop=True)
)

print("\nInterest Rate Data (clean, denormalized):")
print(df_interest_clean.head(10))
print(f"\nShape: {df_interest_clean.shape}")
print(f"\nData types:\n{df_interest_clean.dtypes}")



Interest Rate Data (clean, denormalized):
        date  sweden_interest_rate
0 2021-09-01                -0.104
1 2021-09-02                -0.096
2 2021-09-03                -0.101
3 2021-09-06                -0.109
4 2021-09-07                -0.104
5 2021-09-08                -0.100
6 2021-09-09                -0.110
7 2021-09-10                -0.108
8 2021-09-13                -0.102
9 2021-09-14                -0.100

Shape: (1083, 2)

Data types:
date                    datetime64[ns]
sweden_interest_rate           float32
dtype: object


In [68]:
# Drop missing data from all DataFrames
print("Before dropping missing data:")
print(f"Exchange Rate: {df_exchange_rate_clean.shape[0]} rows, {df_exchange_rate_clean.isnull().sum().sum()} missing values")
print(f"Inflation: {df_inflation_clean.shape[0]} rows, {df_inflation_clean.isnull().sum().sum()} missing values")
print(f"Interest Rate: {df_interest_clean.shape[0]} rows, {df_interest_clean.isnull().sum().sum()} missing values")

df_exchange_rate_clean.dropna(inplace=True)
df_inflation_clean.dropna(inplace=True)
df_interest_clean.dropna(inplace=True)

print("\nAfter dropping missing data:")
print(f"Exchange Rate: {df_exchange_rate_clean.shape[0]} rows")
print(f"Inflation: {df_inflation_clean.shape[0]} rows")
print(f"Interest Rate: {df_interest_clean.shape[0]} rows")

print("\nExchange Rate Data (first 5 rows):")
df_exchange_rate_clean.head()

Before dropping missing data:
Exchange Rate: 2640 rows, 0 missing values
Inflation: 122 rows, 0 missing values
Interest Rate: 1083 rows, 0 missing values

After dropping missing data:
Exchange Rate: 2640 rows
Inflation: 122 rows
Interest Rate: 1083 rows

Exchange Rate Data (first 5 rows):


,date,eur_sek_rate
0,2015-08-28,9.4953
1,2015-08-31,9.5032
2,2015-09-01,9.4954
3,2015-09-02,9.4968
4,2015-09-03,9.3907


## <span style='color:#ff5f27'> 🌍 STEP 8: Prepare DataFrames for Feature Groups </span>

We need to ensure all DataFrames are properly formatted for insertion into Hopsworks Feature Groups:

1. **Exchange Rate**: Should have `date` and `eur_sek_rate` columns
2. **Inflation**: Should have `date`, `country`, and `inflation_rate` columns
3. **Interest Rate**: Should have `date`, `country`, and `interest_rate` columns

All date columns should be datetime64[ns] and numeric columns should be float32.

In [69]:
# Final verification of DataFrames before inserting into Feature Groups

print("=== Exchange Rate DataFrame ===")
print(df_exchange_rate_clean.info())
print(f"\nDate range: {df_exchange_rate_clean['date'].min()} to {df_exchange_rate_clean['date'].max()}")
print(f"\nSample data:")
print(df_exchange_rate_clean.head())

print("\n=== Inflation DataFrame ===")
print(df_inflation_clean.info())
print(f"\nDate range: {df_inflation_clean['date'].min()} to {df_inflation_clean['date'].max()}")
print(f"\nSample data:")
print(df_inflation_clean.head())

print("\n=== Interest Rate DataFrame ===")
print(df_interest_clean.info())
print(f"\nDate range: {df_interest_clean['date'].min()} to {df_interest_clean['date'].max()}")
print(f"\nSample data:")
print(df_interest_clean.head())

=== Exchange Rate DataFrame ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2640 entries, 0 to 2639
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          2640 non-null   datetime64[ns]
 1   eur_sek_rate  2640 non-null   float32       
dtypes: datetime64[ns](1), float32(1)
memory usage: 31.1 KB
None

Date range: 2015-08-28 00:00:00 to 2025-12-16 00:00:00

Sample data:
        date  eur_sek_rate
0 2015-08-28        9.4953
1 2015-08-31        9.5032
2 2015-09-01        9.4954
3 2015-09-02        9.4968
4 2015-09-03        9.3907

=== Inflation DataFrame ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   122 non-null    datetime64[ns]
 1   eu_inflation_rate      122 non-null    float32    

In [70]:
df_aq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2525 entries, 0 to 2524
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     2525 non-null   datetime64[ns]
 1   pm25     2525 non-null   float32       
 2   country  2525 non-null   object        
 3   city     2525 non-null   object        
 4   street   2525 non-null   object        
 5   url      2525 non-null   object        
dtypes: datetime64[ns](1), float32(1), object(4)
memory usage: 128.2+ KB


## <span style='color:#ff5f27'> 🌍 STEP 10: Define Data Validation Rules </span>

We will validate the exchange rate, inflation, and interest rate data before we write them to Hopsworks.

We define data validation rules (expectations in Great Expectations) that ensure:
- **Exchange Rate**: `eur_sek_rate` values are reasonable (typically between 5-15 SEK per EUR)
- **Inflation**: `inflation_rate` values are reasonable (typically between -10% and 50%)
- **Interest Rate**: `interest_rate` values are reasonable (typically between -5% and 20%)

We will attach these expectations to the respective feature groups, so that we validate the data every time we write a DataFrame to the feature group. We want to prevent garbage-in, garbage-out.

### Expectations for Exchange Rate Data

In [71]:
import great_expectations as ge

# Exchange Rate Expectations
exchange_rate_expectation_suite = ge.core.ExpectationSuite(
    expectation_suite_name="exchange_rate_expectation_suite"
)

exchange_rate_expectation_suite.add_expectation(
    ge.core.ExpectationConfiguration(
        expectation_type="expect_column_min_to_be_between",
        kwargs={
            "column": "eur_sek_rate",
            "min_value": 5.0,
            "max_value": 15.0,
            "strict_min": True
        }
    )
)

print("Exchange Rate expectation suite created successfully!")

Exchange Rate expectation suite created successfully!


### Expectations for Inflation Data

Here, we define an expectation for the `inflation_rate` column, where we expect values to be between -10% and 50%.

In [84]:
# Inflation Expectations
inflation_expectation_suite = ge.core.ExpectationSuite(
    expectation_suite_name="inflation_expectation_suite"
)

inflation_expectation_suite.add_expectation(
    ge.core.ExpectationConfiguration(
        expectation_type="expect_column_min_to_be_between",
        kwargs={
            "column": "sweden_inflation_rate",
            "min_value": -10.0,
            "max_value": 50.0,
            "strict_min": True
        }
    )
)

print("Inflation expectation suite created successfully!")

Inflation expectation suite created successfully!


### Expectations for Interest Rate Data

Here, we define an expectation for the `interest_rate` column, where we expect values to be between -5% and 20%.

In [95]:
# Interest Rate Expectations
interest_rate_expectation_suite = ge.core.ExpectationSuite(
    expectation_suite_name="interest_rate_expectation_suite"
)

interest_rate_expectation_suite.add_expectation(
    ge.core.ExpectationConfiguration(
        expectation_type="expect_column_min_to_be_between",
        kwargs={
            "column": "sweden_interest_rate",
            "min_value": -5.0,
            "max_value": 20.0,
            "strict_min": True
        }
    )
)

print("Interest Rate expectation suite created successfully!")

Interest Rate expectation suite created successfully!


---

### <span style="color:#ff5f27;"> 🔮 STEP 11: Connect to Hopsworks and save configuration as a secret</span>

In [74]:
# Connect to Hopsworks Feature Store
project = hopsworks.login()
fs = project.get_feature_store()
secrets = hopsworks.get_secrets_api()

2025-12-17 16:28:07,349 INFO: Closing external client and cleaning up certificates.
Connection closed.
2025-12-17 16:28:07,350 INFO: Initializing external client
2025-12-17 16:28:07,351 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-12-17 16:28:11,299 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1286353


#### Save currency pair configuration as a secret

This will be used later in the daily feature pipeline and batch inference pipeline.

In [75]:
# Configuration for exchange rate prediction
config_dict = {
    "currency_pair": "EUR_SEK",
    "base_currency": "EUR",
    "quote_currency": "SEK",
    "countries": ["EU", "Sweden"]
}

# Convert the dictionary to a JSON string
config_str = json.dumps(config_dict)

# Replace any existing secret with the new value
try:
    secret = secrets.get_secret("EXCHANGE_RATE_CONFIG")
    if secret is not None:
        secret.delete()
        print("Replacing existing EXCHANGE_RATE_CONFIG")
except:
    print("No existing EXCHANGE_RATE_CONFIG found")

secrets.create_secret("EXCHANGE_RATE_CONFIG", config_str)
print("Secret created successfully!")

Replacing existing EXCHANGE_RATE_CONFIG
Secret created successfully, explore it at https://c.app.hopsworks.ai:443/account/secrets
Secret created successfully!


### <span style="color:#ff5f27;"> 🔮 STEP 12: Create the Feature Groups and insert the DataFrames in them </span>

### <span style='color:#ff5f27'> 💱 Exchange Rate Data</span>
    
1. Provide a name, description, and version for the feature group.
2. Define the `primary_key`: Each exchange rate measurement is uniquely identified by `date`.
3. Define the `event_time`: The column that stores the timestamp - `date`.
4. Attach the `expectation_suite` containing data validation rules.

In [76]:
# Create Exchange Rate Feature Group
exchange_rate_fg = fs.get_or_create_feature_group(
    name='exchange_rate',
    description='EUR to SEK exchange rate data',
    version=1,
    primary_key=['date'],
    event_time='date',
    expectation_suite=exchange_rate_expectation_suite
)

#### Insert the Exchange Rate DataFrame into the Feature Group

In [77]:
# Insert exchange rate data
exchange_rate_fg.insert(df_exchange_rate_clean)

2025-12-17 16:28:15,390 INFO: 	1 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868111


Uploading Dataframe: 100.00% |██████████| Rows 2640/2640 | Elapsed Time: 00:02 | Remaining Time: 00:00


Launching job: exchange_rate_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1286353/jobs/named/exchange_rate_1_offline_fg_materialization/executions


(Job('exchange_rate_1_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "eur_sek_rate",
           "min_value": 5.0,
           "max_value": 15.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 796673
         }
       },
       "result": {
         "observed_value": 9.138100624084473,
         "element_count": 2640,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-12-17T08:28:15.000390Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     }
   ],
   "evaluation_parameters": {},
   "statistics": {
     "evaluated_expectations": 1,
 

#### Enter a description for each feature in the Exchange Rate Feature Group

In [78]:
exchange_rate_fg.update_feature_description("date", "Date of the exchange rate measurement")
exchange_rate_fg.update_feature_description("eur_sek_rate", "Exchange rate: EUR to SEK (Swedish Krona)")

### <span style='color:#ff5f27'> 📈 Inflation Data</span>
    
1. Provide a name, description, and version for the feature group.
2. Define the `primary_key`: Each inflation measurement is uniquely identified by `date` and `country`.
3. Define the `event_time`: The column that stores the timestamp - `date`.
4. Attach the `expectation_suite` containing data validation rules.

In [88]:
# Create Inflation Feature Group
inflation_fg = fs.get_or_create_feature_group(
    name='inflation',
    description='Monthly inflation rates for EU and Sweden',
    version=2,
    primary_key=['date'],
    event_time='date',
    expectation_suite=inflation_expectation_suite
)

#### Insert the Inflation DataFrame into the Feature Group

In [89]:
# Insert inflation data
inflation_fg.insert(df_inflation_clean, wait=True)

Feature Group created successfully, explore it at 
https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868118
2025-12-17 16:32:49,153 INFO: 	1 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868118


Uploading Dataframe: 100.00% |██████████| Rows 122/122 | Elapsed Time: 00:02 | Remaining Time: 00:00


Launching job: inflation_2_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1286353/jobs/named/inflation_2_offline_fg_materialization/executions
2025-12-17 16:33:12,552 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2025-12-17 16:33:15,857 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2025-12-17 16:34:55,227 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2025-12-17 16:34:55,512 INFO: Waiting for log aggregation to finish.
2025-12-17 16:35:04,480 INFO: Execution finished successfully.


(Job('inflation_2_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "sweden_inflation_rate",
           "min_value": -10.0,
           "max_value": 50.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 796674
         }
       },
       "result": {
         "observed_value": -0.20000000298023224,
         "element_count": 122,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-12-17T08:32:49.000153Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     }
   ],
   "evaluation_parameters": {},
   "statistics": {
     "evaluated_expectatio

#### Enter a description for each feature in the Inflation Feature Group

In [ ]:
inflation_fg.update_feature_description("date", "Date of the inflation rate measurement (monthly)")
inflation_fg.update_feature_description("country", "Country or region (EU or Sweden)")
inflation_fg.update_feature_description("inflation_rate", "Monthly inflation rate as a percentage")

### <span style='color:#ff5f27'> 💰 Interest Rate Data</span>
    
1. Provide a name, description, and version for the feature group.
2. Define the `primary_key`: Each interest rate measurement is uniquely identified by `date` and `country`.
3. Define the `event_time`: The column that stores the timestamp - `date`.
4. Attach the `expectation_suite` containing data validation rules.

In [97]:
# Create Interest Rate Feature Group
interest_rate_fg = fs.get_or_create_feature_group(
    name='interest_rate',
    description='Interest rates for EU and Sweden',
    version=3,
    primary_key=['date'],
    event_time='date',
    expectation_suite=interest_rate_expectation_suite
)


#### Insert the Interest Rate DataFrame into the Feature Group

In [98]:
# Insert interest rate data
interest_rate_fg.insert(df_interest_clean, wait=True)

Feature Group created successfully, explore it at 
https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868120
2025-12-17 17:00:57,289 INFO: 	1 expectation(s) included in expectation_suite.
Validation failed.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1286353/fs/1273974/fg/1868120


Uploading Dataframe: 100.00% |██████████| Rows 1083/1083 | Elapsed Time: 00:01 | Remaining Time: 00:00


Launching job: interest_rate_3_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1286353/jobs/named/interest_rate_3_offline_fg_materialization/executions
2025-12-17 17:01:24,755 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2025-12-17 17:01:31,332 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2025-12-17 17:02:57,261 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2025-12-17 17:02:57,544 INFO: Waiting for log aggregation to finish.
2025-12-17 17:03:06,499 INFO: Execution finished successfully.


(Job('interest_rate_3_offline_fg_materialization', 'SPARK'),
 {
   "success": false,
   "results": [
     {
       "success": false,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "sweden_interest_rate",
           "min_value": -5.0,
           "max_value": 20.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 796675
         }
       },
       "result": {
         "observed_value": -9.03800106048584,
         "element_count": 1083,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-12-17T09:00:57.000288Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     }
   ],
   "evaluation_parameters": {},
   "statistics": {
     "evaluated_expectat

#### Enter a description for each feature in the Interest Rate Feature Group

In [ ]:
interest_rate_fg.update_feature_description("date", "Date of the interest rate measurement")
interest_rate_fg.update_feature_description("country", "Country or region (EU or Sweden)")
interest_rate_fg.update_feature_description("interest_rate", "Interest rate as a percentage")